In [23]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import GradientBoostingRegressor
import warnings
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

warnings.filterwarnings('ignore')

train = pd.read_csv('/kaggle/input/california-homelessness-prediction-challenge/train.csv')
test = pd.read_csv('/kaggle/input/california-homelessness-prediction-challenge/test.csv')
sample_sub = pd.read_csv('/kaggle/input/california-homelessness-prediction-challenge/sample_submission.csv')

print(f"Train shape: {train.shape}, Test shape: {test.shape}")
print(f"Columns: {train.columns.tolist()}")
print("\nTarget stats:", train['HOMELESS_RATE'].describe())

numeric_cols = train.select_dtypes(include=np.number).columns
corr = train[numeric_cols].corrwith(train['HOMELESS_RATE']).sort_values(ascending=False)
print("\nTop 10 correlated features:\n", corr.head(10))

X = train.drop(columns=['HOMELESS_RATE', 'ID'])
y = train['HOMELESS_RATE']
X_test = test.drop(columns=['ID'])

le = LabelEncoder()
for col in X.select_dtypes(include=['object']).columns:
    X[col] = le.fit_transform(X[col].astype(str))
    X_test[col] = le.transform(X_test[col].astype(str))

for col in X.columns:
    if X[col].dtype in ['int64', 'float64']:
        upper = X[col].quantile(0.99)
        lower = X[col].quantile(0.01)
        X[col] = np.clip(X[col], lower, upper)
        X_test[col] = np.clip(X_test[col], lower, upper)

scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)

gb = GradientBoostingRegressor(n_estimators=200, learning_rate=0.05, random_state=42)
gb.fit(X_scaled, y)

test_preds = gb.predict(X_test_scaled)
submission = pd.DataFrame({'ID': test['ID'], 'HOMELESS_RATE': test_preds})
submission.to_csv('submission.csv', index=False)
print("\nSubmission saved!")

Train shape: (130, 33), Test shape: (56, 32)
Columns: ['ID', 'HOMELESS_RATE', 'AGE_U18_PCT', 'AGE_18_24_PCT', 'AGE_25_34_PCT', 'AGE_35_44_PCT', 'AGE_45_54_PCT', 'AGE_55_59_PCT', 'AGE_60_61_PCT', 'AGE_62_64_PCT', 'AGE_65_69_PCT', 'AGE_70_79_PCT', 'AGE_80_PLUS_PCT', 'AGE_25_PLUS_PCT', 'FAMILY_MEMBERS_UNDER_18_PCT', 'RACE_WHITE_NH_PCT', 'RACE_BLACK_NH_PCT', 'RACE_NATIVE_NH_PCT', 'RACE_ASIAN_NH_PCT', 'RACE_PACIFIC_NH_PCT', 'RACE_TWO_OR_MORE_NH_PCT', 'RACE_HISPANIC_ANY_PCT', 'VETERAN_POP_PCT', 'NONVETERAN_POP_PCT', 'DISABILITY_POP_PCT', 'NODISABILITY_POP_PCT', 'TOTAL_HOUSEHOLDS_PCT', 'FAMILY_HH_TOTAL', 'FAMILY_HH_CHILD_LT18_PCT', 'NONFAMILY_SINGLE_MALE_PCT', 'NONFAMILY_SINGLE_FEMALE_PCT', 'MULTI_PERSON_NONFAMILY_HH_PCT', 'INDIVIDUALS_NOT_IN_FAMILY_UNITS_PCT']

Target stats: count    130.000000
mean       0.003736
std        0.006679
min        0.000000
25%        0.000615
50%        0.001652
75%        0.003839
max        0.058798
Name: HOMELESS_RATE, dtype: float64

Top 10 correlated featu

In [24]:
test.head()

,ID,AGE_U18_PCT,AGE_18_24_PCT,AGE_25_34_PCT,AGE_35_44_PCT,AGE_45_54_PCT,AGE_55_59_PCT,AGE_60_61_PCT,AGE_62_64_PCT,AGE_65_69_PCT,AGE_70_79_PCT,AGE_80_PLUS_PCT,AGE_25_PLUS_PCT,FAMILY_MEMBERS_UNDER_18_PCT,RACE_WHITE_NH_PCT,RACE_BLACK_NH_PCT,RACE_NATIVE_NH_PCT,RACE_ASIAN_NH_PCT,RACE_PACIFIC_NH_PCT,RACE_TWO_OR_MORE_NH_PCT,RACE_HISPANIC_ANY_PCT,VETERAN_POP_PCT,NONVETERAN_POP_PCT,DISABILITY_POP_PCT,NODISABILITY_POP_PCT,TOTAL_HOUSEHOLDS_PCT,FAMILY_HH_TOTAL,FAMILY_HH_CHILD_LT18_PCT,NONFAMILY_SINGLE_MALE_PCT,NONFAMILY_SINGLE_FEMALE_PCT,MULTI_PERSON_NONFAMILY_HH_PCT,INDIVIDUALS_NOT_IN_FAMILY_UNITS_PCT
0,AL_13,0.342169,0.088348,0.147178,0.134551,0.144855,0.073641,0.032186,0.036751,0.097467,0.157331,0.088014,0.342812,0.342169,0.203063,0.101112,0.001817,0.346941,0.013201,0.041719,0.287720,0.033474,0.794763,0.494866,0.019192,0.228051,0.161873,0.062935,0.098939,0.066178,0.022780,0.066178
1,LA_10,0.370642,0.090580,0.197082,0.152119,0.127954,0.060101,0.026158,0.032274,0.088054,0.103299,0.065468,0.256822,0.370642,0.122154,0.199022,0.001841,0.166771,0.001335,0.028428,0.475139,0.017774,0.796692,0.496151,0.025846,0.211684,0.121713,0.046176,0.075537,0.089971,0.030425,0.089971
2,SD_15,0.462069,0.074806,0.135474,0.148376,0.121019,0.064783,0.028337,0.037049,0.110785,0.140659,0.066802,0.318246,0.462069,0.670616,0.012118,0.003135,0.048775,0.001748,0.062642,0.198885,0.082553,0.676585,0.484905,0.040998,0.250991,0.191846,0.085025,0.106820,0.059145,0.018705,0.059145
3,SB_11,0.324003,0.070298,0.124383,0.123496,0.137429,0.089677,0.036605,0.056111,0.136795,0.175807,0.087397,0.400000,0.324003,0.698923,0.025586,0.008360,0.005953,0.000000,0.022039,0.239139,0.051425,0.784927,0.465484,0.023813,0.274098,0.193920,0.053325,0.140595,0.080177,0.022799,0.080177
4,SB_09,0.443576,0.091267,0.133863,0.124290,0.142770,0.074939,0.034057,0.045485,0.097654,0.117705,0.047722,0.263081,0.443576,0.243822,0.041137,0.001388,0.387357,0.000198,0.036309,0.288953,0.029695,0.747554,0.510769,0.028775,0.267414,0.216761,0.088704,0.128057,0.050654,0.020307,0.050654


In [29]:
pd.set_option("display.max_columns",None)
df=pd.read_csv("/kaggle/input/california-homelessness-prediction-challenge/train.csv")
test=pd.read_csv("/kaggle/input/california-homelessness-prediction-challenge/test.csv")
ID=test.ID
test.drop(columns=["ID"],axis=1,inplace=True)
print(f"data shape: {df.shape}")
print(f"Check null values: {df.isnull().sum()}")
df.drop(columns=["ID"],axis=1,inplace=True)

X = df.drop(columns=['HOMELESS_RATE'])
y = df['HOMELESS_RATE']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

rf = RandomForestRegressor(n_estimators=1000, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)

rmse = mean_squared_error(y_test, y_pred, squared=False)
r2 = r2_score(y_test, y_pred)

print(f"RMSE: {rmse:.6f}")
print(f"R² Score: {r2:.4f}")

rf = RandomForestRegressor(n_estimators=20000, max_depth=20, random_state=42, n_jobs=-1)
rf.fit(X, y)

test_preds = rf.predict(test)
submission = pd.DataFrame({'ID': ID, 'HOMELESS_RATE': test_preds})
submission.to_csv('r_submission.csv', index=False)
print("\nSubmission saved!")

data shape: (130, 33)
Check null values: ID                                     0
HOMELESS_RATE                          0
AGE_U18_PCT                            0
AGE_18_24_PCT                          0
AGE_25_34_PCT                          0
AGE_35_44_PCT                          0
AGE_45_54_PCT                          0
AGE_55_59_PCT                          0
AGE_60_61_PCT                          0
AGE_62_64_PCT                          0
AGE_65_69_PCT                          0
AGE_70_79_PCT                          0
AGE_80_PLUS_PCT                        0
AGE_25_PLUS_PCT                        0
FAMILY_MEMBERS_UNDER_18_PCT            0
RACE_WHITE_NH_PCT                      0
RACE_BLACK_NH_PCT                      0
RACE_NATIVE_NH_PCT                     0
RACE_ASIAN_NH_PCT                      0
RACE_PACIFIC_NH_PCT                    0
RACE_TWO_OR_MORE_NH_PCT                0
RACE_HISPANIC_ANY_PCT                  0
VETERAN_POP_PCT                        0
NONVETERAN_POP_P